# Data Cleaning Plan — Financial Manager AI

## Objective

Convert completed inventory and EDA findings into explicit, auditable cleaning and standardization rules **before any transformation is performed**. This notebook contains documentation only: no executable cells, dataset loading, cleaning, processed exports, models, database work, FX logic, or application implementation.

**Status: proposed plan, awaiting user approval.** “Required” describes a prerequisite for a future approved cleaning run, not authorization to execute it now. Representation proposals are not new facts about the data. Dataset-specific rules remain separate from Financial Manager conventions.

The plan follows ten principles: raw data is immutable; cleaning is reproducible; original values remain traceable; missingness does not justify automatic removal; extremes are not automatically errors; technical and semantic types differ; financial meaning must survive transformations; decisions cite EDA evidence; currency is never silently assigned; and source-specific rules do not become universal business rules.

### Evidence Register

Only the following existing notebooks and their saved outputs support this plan. Section references are used instead of mutable execution counts. No external source search or new EDA is performed.

| Reference | Notebook and sections | Evidence used |
|---|---|---|
| I-RG | [Inventory](01_dataset_inventory.ipynb), Receiver General §§2–14 | All 16 fields/types; missing counts; source documentation review; fiscal and granularity limits; candidate key |
| I-CF | [Inventory](01_dataset_inventory.ipynb), Company Financials §§2–14 | All 16 fields/types; whitespace; eight formatted measures; no company ID; empirical key |
| RG-B | [Receiver General EDA](02_eda_receiver_general.ipynb), §§1–2, final checks | 9,282 × 16; 29,876 null cells; no complete duplicates/date failures/key conflicts; preservation passes |
| RG-D | [Receiver General EDA](02_eda_receiver_general.ipynb), §§3–7, 10 | Skew; debit/credit and category differences; sparse accounting dimensions; concentration |
| RG-T | [Receiver General EDA](02_eda_receiver_general.ipynb), §§8–9 | Effective-date calendar coverage; partial endpoint; temporal profiles |
| RG-M | [Receiver General EDA](02_eda_receiver_general.ipynb), §§11, 14 | Category-dependent missingness; field-presence patterns; summary-line relationships |
| RG-X | [Receiver General EDA](02_eda_receiver_general.ipynb), §§12–18 | Global/peer extremes; peer feasibility; quality issues; unchanged capability assessments |
| CF-B | [Company Financials EDA](03_eda_company_financials.ipynb), §§1–5, final checks | 700 × 16; no raw nulls/complete duplicates; candidate key; monthly labels; no company ID |
| CF-P | [Company Financials EDA](03_eda_company_financials.ipynb), §3 | Parsing audit: 5,542 successful measure cells, 58 unresolved dashes, no unrecognized formats |
| CF-D | [Company Financials EDA](03_eda_company_financials.ipynb), §§6–10 | Distributions and category scales; 58 negative Profit records; no valid company growth |
| CF-R | [Company Financials EDA](03_eda_company_financials.ipynb), §§11–12 | 2,042 testable arithmetic checks within 0.011; conditional ratios and correlations |
| CF-M | [Company Financials EDA](03_eda_company_financials.ipynb), §13 | All 53 Discounts dashes in None; all 5 Profit dashes in Enterprise/Low; zero hypothesis unconfirmed |
| CF-X | [Company Financials EDA](03_eda_company_financials.ipynb), §§14–15 | Global/peer disagreement; seven pricing groups; small period-level groups |
| CF-C | [Company Financials EDA](03_eda_company_financials.ipynb), §§16–20 | Unknown currency; unresolved semantics/provenance; quality and capability conclusions |

**Action vocabulary:** KEEP retains meaning/value; PARSE creates a separate typed value; STANDARDIZE changes a documented representation only; FLAG records a condition without repairing it; VALIDATE checks an invariant; PRESERVE AS-IS forbids semantic alteration; DEFER requires further evidence or approval. Multiple actions may apply to one field. All are proposals.

## 1. Cleaning Principles

| Area | Proposed project-wide principle |
|---|---|
| Raw preservation | Immutable source files, content hashes, and immutable raw DataFrames; work only on separate copies. Never overwrite a raw path. |
| Reproducibility | Record source hashes, rule/version identifiers, environment versions, parser settings, header mapping, and every exception. A repeated run of the same rules/input must produce equivalent data and flags. |
| Auditability | Preserve a source-file reference and original row position, plus original tokens for parsed/standardized fields. A row position is lineage, not a business key. Retain before/after values, rule ID, and reason for any approved change. |
| Semantic typing | Identifiers and codes remain categorical even when Pandas inferred a numeric dtype. Dates, period labels, quantities, counts, prices, and monetary amounts retain distinct meanings. |
| Missingness | Preserve rows and sparse columns. Distinguish source null, unresolved placeholder, parser failure, and unassessed applicability. No blanket null-row removal or imputation. |
| Duplicates | Test full rows and candidate keys separately, before and after standardization. Repeated references need not be duplicate business events. No deletion is currently justified. |
| Extremes | Preserve zeros, negatives, large values, and rare categories. Descriptive thresholds may create review evidence, never automatic invalidity. |
| Identifiers | Prefer exact source-token strings. Never reconstruct lost digits or add padding from guesses. Preserve alphanumeric values and leading zeros that actually exist. |
| Time | Parse only explicit supported formats, retain labels, and distinguish date/period from payment date. No inferred fiscal-to-calendar rewrite or invented observation for a gap. |
| Money | Preserve source sign, magnitude, and scale; propose exact decimal parsing from tokens. No silent rounding, sign netting, unit rescaling, balancing, or aggregation into new facts. |
| Currency | Retain confirmed/inferred/unknown status and evidence separately from amounts. Unknown stays unknown. No default BRL/USD/CAD and no FX. |
| Categories | Use an explicit header map; trim only approved outer whitespace in separate representations. Preserve raw tokens. No automatic case folding, translation, code merging, or guessed label meanings. |

**RAW DATA** is the unchanged original file and its faithful loaded representation. **CLEANED DATA** would be a traceable, row-preserving representation with validated types, retained originals, statuses, and limitations; “cleaned” does not mean semantically complete or operationally ready. **ANALYTICAL / DERIVED FEATURES** include ratios, log displays, fiscal interpretations, growth, normalized measures, peer scores, and statistical thresholds. These are not base cleaned facts and are not production features authorized by this plan. Optional descriptive review flags belong in an audit report with their method, not as rewritten amounts.

## 2. Receiver General — Cleaning Requirements

Baseline: **9,282 rows, 16 fields**. Current dtypes are the saved Pandas inventory types, not a guarantee of original CSV lexical form. Field labels below abbreviate the exact bilingual headers in I-RG; a future implementation must keep a one-to-one mapping to those exact headers.

**Original retention:** retain the source token for every field below (directly or through immutable row/field lineage); include companion raw values for all parsed or standardized fields. No field is dropped. Null counts refer to the inventory/EDA baseline.

| Rule / Field | Current state / semantic type | Issue | Proposed rule and cleaned representation | Action | Evidence |
|---|---|---|---|---|---|
| RG01 Accounting Control Number | int64; grouping identifier; 0 null | Repeats; 148 occurrences after first | Exact token string `accounting_control_number`; validate joint key, not individual uniqueness | PARSE; VALIDATE | I-RG §11; RG-B |
| RG02 Journal Voucher Type Code | str; accounting category; 26 null | 39 known codes; differing distributions | Nullable string `journal_voucher_type_code`; retain null and exact code; no invented business labels | KEEP; FLAG | I-RG; RG-D, RG-M |
| RG03 Fiscal Year | str; fiscal-period label; 0 null | Local effective-date alignment unresolved | String `fiscal_year`, preserving supplied label; syntax check without date reassignment | PRESERVE AS-IS; VALIDATE; DEFER | I-RG §9; RG-T |
| RG04 Journal Voucher Identifier | float64; optional identifier; 9,104 null | 98.08% absent; float is not semantic type | Nullable exact source string `journal_voucher_identifier`; preserve raw; no integer reconstruction from float | PARSE; FLAG | I-RG; RG-M |
| RG05 Accounting Effective Date | str; date; 0 null; all parse | Not verified cash/payment date | Date-only `accounting_effective_date` parsed strictly as YYYY-MM-DD, with raw text | PARSE; VALIDATE | RG-B, RG-T |
| RG06 Department Number | float64; organizational code; 26 null | Identifier, not a measure | Nullable source string `department_number`; no guessed names or zero padding | PARSE; FLAG | I-RG; RG-D |
| RG07 Fiscal Month | int64; fiscal dimension; 0 null | Not a calendar month | Nullable integer `fiscal_month` as categorical period label; validate 1–12; retain original | KEEP; VALIDATE; DEFER | I-RG §9 |
| RG08 Journal Voucher Item Identifier | int64; within-voucher sequence identifier; 0 null | Not independently unique | Exact string `journal_voucher_item_identifier`; preserve sequence meaning; validate joint key | PARSE; VALIDATE | I-RG §§4,11; RG-B |
| RG09 Financial Coding Block | str; accounting code; 9,000 null | 96.96% absent; alphanumeric values occur | Nullable string `financial_coding_block`; retain full block and null; no automatic decomposition or inference | KEEP; FLAG | RG-D §7; RG-M |
| RG10 Subledger Account Identifier | float64; account code; 2,186 null | 23.55% of rows missing, carrying 80.56% of recorded magnitude | Nullable exact source string `subledger_account_identifier`; unknown is not an account category | PARSE; FLAG | RG-D §10; RG-M |
| RG11 Credit/Debit Code | str; direction category; 0 null | DR/CR are not expense/revenue labels | String `credit_debit_code`; retain DR/CR; validate observed vocabulary, flag new codes rather than remap | KEEP; VALIDATE | RG-D §4 |
| RG12 Journal Voucher Item Amount | float64; monetary magnitude; 0 null | Strong skew, 2 zeros; currency/scale context unresolved | Exact decimal `journal_voucher_item_amount` from source token; preserve raw sign/scale; no sign derived from DR/CR | PARSE; VALIDATE; FLAG | RG-D §3; RG-X §12 |
| RG13 Control Data Type | str; control category; 178 null | Type-dependent coverage | Nullable string `control_data_type`; preserve codes/nulls; no automatic row taxonomy | KEEP; FLAG | RG-M |
| RG14 Accounting Summary Transaction Line Count | float64; grouped-detail count; 26 null | Counts 1–98 among populated values; mixed record context | Nullable integer `accounting_summary_transaction_line_count` only after exact integrality validation; never fill with 1 | PARSE; VALIDATE; FLAG | RG-D §7; I-RG §4 |
| RG15 GBS Control Number | float64; system control identifier; 239 null | Fidelity must be protected; not a measure | Nullable exact source string `gbs_control_number`; no float-to-int conversion as source of truth | PARSE; VALIDATE; FLAG | I-RG; RG-X §16 |
| RG16 General Ledger Account Code | str; account classification; 9,091 null | 97.94% absent; values include alphanumeric code 6B000 | Nullable string `general_ledger_account_code`; preserve alphabetic characters/nulls | KEEP; FLAG | RG-D §7 |

Identifier token recovery and decimal materialization are future validation tasks, not assertions that the original source is damaged. Unexpected lexical forms must remain raw and be reported rather than silently reformatted.

## 3. Receiver General — Temporal Fields

| Field | Proposed handling | Boundary |
|---|---|---|
| Accounting Effective Date | Strict YYYY-MM-DD parsing to a date-only representation; preserve source text and report parsing status | Baseline 188 distinct dates, April 3, 2023–January 3, 2024; no timezone or payment date is invented |
| Fiscal Year | Retain the label as supplied; validate expected year-pair syntax using the inventory examples and documented field meaning | Do not overwrite it using the effective-date year |
| Fiscal Month | Retain the 1–12 fiscal label as an integer dimension | Do not rename as calendar month or derive calendar dates from it |

I-RG §9 records the official April–March Receiver General convention from a related publication, but the exact relationship of local fiscal labels to effective dates remains unresolved. Neither that convention nor matching counts proves which local records are prior-year adjustments. Preserve both fields independently and carry a **dataset-level fiscal-alignment-unresolved status**, not a row-level assertion of mismatch.

RG-T shows a partial January endpoint and dates absent from the extract. Do not synthesize zero-value dates, infer economic inactivity, or fill periods. Calendar month/week groupings may later be regenerated as analytical variables from the parsed effective date; they are not replacements for fiscal labels.

## 4. Receiver General — Identifiers and Codes

All fields in RG01, RG04, RG06, RG08, RG10, and RG15 are identifiers, even if currently int64/float64. RG02, RG09, RG11, RG13, and RG16 are categorical/accounting codes. They must not become continuous model variables through numeric dtype alone.

**Proposed conversion contract:** obtain original CSV tokens through a separate source-preserving read in the future cleaning notebook. Retain both token and nullable string representation. Do not stringify previously rounded float values and claim original fidelity. Preserve observed leading zeros and alphanumerics; do not manufacture fixed widths. If a source token contains decimal/exponent notation, validate its exact meaning first; do not generically strip `.0`, round, or infer lost digits. Record any unresolved representation as an exception while retaining the row.

**Candidate key:** Accounting Control Number + Journal Voucher Item Identifier is complete and unique in this extract. Revalidate before and after any representation change, recording newly colliding values. The control number alone repeats and may legitimately group records. The candidate does not prove uniqueness across future files, periods, or underlying business events and is not a database primary key decision.

**Row lineage proposal:** source content hash + source row position can locate an imported row for audits. It must remain distinct from a business identifier or proof of deduplication. Do not join the two datasets using superficially similar numbers, periods, or amounts. Evidence: I-RG §§4,11; RG-B; RG-X.

## 5. Receiver General — Missing Values

Classifications below are **hypotheses or documented possibilities**, not a completed record-type classifier. “Potentially invalid” requires a verified mandatory-field rule; current blanks are not assigned that label merely because they are absent.

| Field(s) | Evidence / classification | Proposed treatment | Analysis restriction |
|---|---|---|---|
| Voucher Identifier; Control Data Type | Strong category/field-presence association; likely process-related, documentation-dependent | Keep null and record source-null status; investigate applicability | No inferred voucher/control identity or universal completion rule |
| General Ledger Account; Coding Block | Very sparse, category-dependent; structural explanation plausible, not verified for every row | Keep null and field; investigate source/type applicability | Use populated subsets only with explicit coverage and exclusions; do not claim representativeness |
| Subledger Identifier | 2,186 absent; 80.56% of recorded magnitude has no label; category-dependent | Keep/flag null; do not assign an unknown pseudo-account | Account-specific analyses exclude unavailable labels only in an explicitly reported view, never from the base cleaned dataset |
| Department; Summary Line Count | 26 absent each; dictionary permits type-dependent blanks | Keep/flag null; documentation-dependent structural possibility | No zero department or default line count; no automatic grouped-detail reconstruction |
| Voucher Type | 26 absent; unresolved relationship to record type | Keep null; investigate; do not derive it from other codes | Peer comparisons requiring this code report excluded rows |
| GBS Control Number | 239 absent; cause remains unexplained/documentation-dependent | Keep/flag null; investigate lineage | No generated source-system ID |
| Currently complete core fields | Control number, item identifier, date, fiscal fields, direction, amount | Validate the baseline; any new absence is a source/schema-change investigation | A future required-field contract must be explicit; no arbitrary mandatory label for sparse optional fields |

Source-null counts total **29,876** and should remain reconcilable by field. Analysis exclusions are scoped filters with counts, not deletions from cleaned outputs. No blanket `dropna()`, interpolation, mean/median fill, category-mode fill, or inference from correlated fields. Evidence: I-RG §6; RG-B; RG-D; RG-M.

## 6. Receiver General — Extreme Values

Preserve all monetary observations, including the two zeros and values reaching approximately 39.56 billion. The strong right tail (sample skewness approximately 10.10) and type/department concentration are not invalidity criteria. Global and peer IQR screens disagree; EDA reported 2,194 global upper-fence candidates versus 937 peer candidates within its eligible population. These are investigation counts, not error counts or approved production cutoffs. Evidence: RG-D §§3–6; RG-X §12.

Future descriptive review may consider DR/CR, voucher type, department, Control Data Type, covered account fields, effective-date calendar period, and summary-line context. Missing accounts and very small peers must be disclosed. Do not use sparse ledger coverage as if universal, treat DR as expense/CR as revenue, classify concentration as risk, or turn mixed summary/item amounts into cash flows.

**Proposed default:** KEEP monetary values; optionally FLAG a contextual review candidate with method, threshold, population, sample size, and reason. Do not winsorize, rescale, delete, net signs, divide summary amounts into fictional transactions, or overwrite extremes. A value change/removal would require independent source evidence, a separately approved corrective rule, and a retained audit trail. No such correction is justified by the current EDA.

## 7. Company Financials — Cleaning Requirements

Baseline: **700 rows, 16 fields, no raw nulls**. This is sales-record data, not an identified company financial-statement panel. Every original field is retained through raw companions/lineage. No company/entity ID, industry mapping, or currency can be created from the labels.

| Rule / Field | Current state / semantic type | Issue | Proposed rule and cleaned representation | Action | Evidence |
|---|---|---|---|---|---|
| CF01 Segment | str; commercial category; 0 null | Not a company or established industry | `segment` nullable string; preserve label/case; outer-whitespace audit only | KEEP; VALIDATE | CF-B §§4–5 |
| CF02 Country | str; geographic category; 0 null | Not currency/entity identity | `country` string; keep original naming; no ISO-country mapping required now | KEEP; PRESERVE AS-IS | CF-B; CF-C |
| CF03 Product | str; product category; 0 null; outer spaces | Formatting is not identity | `product` string; proposed outer trim in separate value, with raw token and collision report | STANDARDIZE; VALIDATE | I-CF §5; CF-B |
| CF04 Discount Band | str; category; 0 null; outer spaces | Literal None is meaningful; no dictionary thresholds | `discount_band` string; outer trim; retain None as text, not null | STANDARDIZE; VALIDATE | CF-M; CF-R |
| CF05 Units Sold | str; quantity; 700 numeric interpretations | Dollar formatting and fractional values | `units_sold` exact decimal quantity under documented parser; preserve raw, units status unknown; do not cast to integer | PARSE; FLAG | CF-P; CF-C |
| CF06 Manufacturing Price | str; price; 700 parses | Monetary units/currency unconfirmed | `manufacturing_price` exact decimal plus raw/parse status; no aggregation into costs | PARSE; VALIDATE | CF-P; CF-D |
| CF07 Sale Price | str; price and candidate-key component; 700 parses | Parsed canonical values may affect key equality | `sale_price` exact decimal plus raw/status; revalidate key before/after parsing | PARSE; VALIDATE | I-CF §11; CF-P |
| CF08 Gross Sales | str; monetary amount; 700 parses | Grouping separators, unknown currency | `gross_sales` exact decimal plus raw/status | PARSE; VALIDATE | CF-P; CF-R |
| CF09 Discounts | str; amount; 647 parses, 53 dashes | Dashes not proven zeros | `discounts` nullable exact decimal; dashes remain unresolved with raw token/status | PARSE; FLAG; DEFER | CF-P; CF-M |
| CF10 Sales | str; amount; 700 parses | Header has extra spaces; meaning not company revenue | `sales` exact decimal plus raw/status; do not relabel as revenue | PARSE; STANDARDIZE | I-CF; CF-P; CF-R |
| CF11 COGS | str; cost amount; 700 parses | Financial definition needs source confirmation | `cogs` exact decimal plus raw/status; no payable/cash interpretation | PARSE; VALIDATE | CF-P; CF-R |
| CF12 Profit | str; amount; 695 parses, 5 dashes, 58 parenthesized negatives | Sign convention provisional; losses not errors | `profit` nullable exact decimal under approved parenthesis convention, with raw/status; no inferred zeros | PARSE; FLAG; DEFER | CF-P; CF-M; CF-X |
| CF13 Date | str; monthly marker; 700 successful date parses | Not individual transaction date | `period_date` strict DD/MM/YYYY parsed date with raw text; document monthly granularity | PARSE; VALIDATE | CF-B §§2,5 |
| CF14 Month Number | int64; temporal dimension; 0 null | Redundant label, not financial measure | `month_number` integer 1–12; cross-check parsed date without overwriting conflicts | KEEP; VALIDATE | CF-B |
| CF15 Month Name | str; temporal label; 0 null; outer spaces | Formatting; redundancy retained | `month_name` string with audited outer trim; validate against date/number; preserve raw | STANDARDIZE; VALIDATE | I-CF §9; CF-B |
| CF16 Year | int64; temporal label; 0 null | No fiscal definition | `year` integer; validate against period date; do not rename fiscal_year | KEEP; VALIDATE | CF-B |
| Absent company/entity identifiers | Not supplied | No company-panel key, size, or growth | No fabricated entity; dataset-level `entity_scope_unknown` status | DEFER; FLAG | CF-B §4; CF-D §10 |
| Absent currency information | Not supplied | Dollar symbol is ambiguous | No currency assignment; unknown metadata per §11 | DEFER; FLAG | CF-C §16 |

All field names are proposals, not implemented renames. The original header map must preserve the exact spelling/whitespace. Before trimming values, compare original and proposed category/key counts and report collisions; do not silently merge distinct source tokens.

## 8. Company Financials — Financial String Parsing

**Rule CF-PARSE:** a future parser must validate the entire token under a named, versioned, dataset-specific grammar before interpreting formatting. Use the EDA convention provisionally, subject to plan approval; do not generalize it to other financial sources.

1. Preserve the exact original token, including spaces, symbol, commas, parentheses, and dashes (conceptually `sales_raw` alongside `sales`; not new files now).
2. Distinguish source absence, dollar-dash placeholder, ordinary numeric token, and unrecognized token before numeric interpretation. Literal category None is outside this monetary parser.
3. Accept the observed optional outer whitespace and `$` formatting, a decimal point with two fractional digits, and validated ordinary three-digit or Indian-style two/three-digit comma grouping. Remove grouping separators only after validation. Do not accept arbitrary punctuation or strip all nonnumeric characters.
4. Interpret correctly matched parentheses as negative only under the explicit provisional sign rule used in EDA. Preserve that rule and source uncertainty in metadata. Do not remove genuine minus meaning, clamp negatives, or assume any unexpected sign convention.
5. Parse accepted magnitudes directly into exact decimal values from tokens, never through binary float. Preserve supplied precision; do not quantize to an assumed currency minor unit. Units Sold remains fractional-capable quantity, not money.
6. Leave dollar-dash values without a numeric interpretation: nullable parsed value + `unresolved_placeholder` status + raw token. Do not use arithmetic identities to fill zeros.
7. Unrecognized formats receive `parsing_failed` with field/row/raw/reason. Retain the row and raw value; no zero, mean, guessed percentage, or valid-looking numeric coercion. Unexpected failures block publication until reviewed; they do not authorize deletion.

| Measure | Observed representation / original dtype | Proposed typed result and baseline coverage | Negative / missing policy |
|---|---|---|---|
| Units Sold | str; e.g. `$1,618.50`; dollar/comma formatting | Exact decimal quantity; 700/700 successful | No negatives observed; no integer rounding; future unexpected signs reviewed |
| Manufacturing Price | str; e.g. `$3.00` | Exact decimal price; 700/700 | No unresolved tokens/negatives observed; preserve valid value |
| Sale Price | str; e.g. `$20.00` | Exact decimal price; 700/700 | Preserve source scale/raw key component |
| Gross Sales | str; e.g. `$32,370.00`, larger grouped values | Exact decimal amount; 700/700 | No negative or unresolved tokens observed |
| Discounts | str; numeric tokens and `$-` | Exact decimal or null-with-status; 647/700 numeric | 53 unresolved placeholders; not zero |
| Sales | str; e.g. `$32,370.00` | Exact decimal amount; 700/700 | No negative or unresolved tokens observed |
| COGS | str; e.g. `$16,185.00` | Exact decimal amount; 700/700 | No negative or unresolved tokens observed |
| Profit | str; numeric tokens, `$(40,617.50)`, `$-` | Exact decimal or null-with-status; 695/700 numeric | Preserve 58 negatives under stated convention; 5 unresolved placeholders |

CF-P found **5,542 successful numeric interpretations out of 5,600 measure cells**, 58 unresolved placeholders, and **0 unrecognized formats**. Percent symbols and other inspected currency symbols were absent. Unsupported new forms must be reported, not accommodated through guessed rules. Expected semantic uncertainty is different from parser failure.

A changed decimal representation must reconcile with the EDA numeric interpretation at source precision, while acknowledging EDA floating-point artifacts. Arithmetic validation tolerances are not permission to alter source amounts.

## 9. Company Financials — Temporal Consistency

Propose strict DD/MM/YYYY parsing, then verify agreement with the supplied Year, Month Number, and trimmed Month Name. All 700 date tokens matched first-of-month markers in EDA; expected labels span **September 2013–December 2014**, 16 consecutive periods. A mismatch in a future run is a reported exception, not permission to regenerate or overwrite the supplied fields. Retain first-of-month dates as period markers, not fabricated sale dates.

**Company-period uniqueness cannot be tested** because company identity is absent. Do not invent it from country, segment, or product. The existing six-field candidate is Segment + Country + Product + Discount Band + Date + Sale Price, unique in this extract; the five-field combination without price repeats 12 times. Recheck after trimming/parsing, preserving original key tokens.

There are no dataset-wide missing months inside the observed interval, but only **2 of 150** segment/country/product combinations cover every month; category histories contain 2–16 periods. This is observed absence, not proven missing transactions or missing company history. Keep gaps and uneven counts; no forward-fill, interpolation, synthetic rows, or zero-activity imputation. A coverage report may flag an absent category-period with its explicit expected range, but not label it erroneous. Evidence: CF-B §§2,4–5; CF-D §§9–10.

## 10. Company Financials — Financial Consistency

| Check | Classification | Evidence | Proposed response |
|---|---|---|---|
| Parsed value conforms to approved token grammar and retains magnitude/sign/scale | Hard validation of representation | CF-P; preservation checks | A conversion mismatch or silent coercion blocks publication; preserve source |
| Units Sold × Sale Price approximately equals Gross Sales | Soft validation | 700/700 complete checks within EDA tolerance | Report residual, operands, testability, tolerance; never reconstruct a measure |
| Gross Sales − Discounts approximately equals Sales | Soft validation | 647/647 testable checks; maximum residual 0.01; 53 untestable | Preserve amounts; do not infer discount zeros or balance residuals |
| Sales − COGS approximately equals Profit | Soft validation | 695/695 testable checks; 5 untestable | Preserve signs; report residuals; no derived Profit fill |
| Dash tokens may represent zero | Investigation only | All 53 Discount-dash records have Gross Sales = Sales; all 5 Profit-dash records have Sales = COGS within tolerance | Obtain source convention; leave null-with-status until then |
| Profit/Sales and Discounts/Gross Sales ratios | Investigation only; analytical outputs, not cleaning rules | CF-R conditional ratios with compatible-unit assumptions | No production feature or ratio repair; no ratio-based monetary correction |

The **0.011 source-unit tolerance** belongs to the EDA's two-decimal/floating-point consistency check; retain it only as a named preliminary soft-check parameter if approved. It is not a currency rounding rule, accepted monetary modification, or universal materiality threshold. Exact decimal residuals should be reported; source rounding/business tolerance remains unresolved.

Arithmetic regularity supports investigation but does not prove financial definitions or shared currencies. No Assets = Liabilities + Equity check is applicable: those fields do not exist. Price × quantity must not be replaced with manufacturing-price assumptions. Negative Profit is not a hard failure. Evidence: CF-R; CF-M; CF-C.

## 11. Multi-Currency Cleaning Policy

**Preliminary global policy:** preserve multiple currencies without conversion. The project's earlier provisional BRL scope must not be used as a default for these sources. This plan accommodates currency metadata without changing the planning document, deciding a base currency, or implementing FX.

| Status | Meaning / proposed handling |
|---|---|
| CONFIRMED | An explicit field or authoritative source statement identifies denomination and applicability to the specific amount/record/dataset. Preserve the original declaration, source reference, and validated code. |
| INFERRED | A later, explicitly approved hypothesis proposes denomination without direct confirmation. Retain proposed code separately, rationale, evidence, and categorical confidence status; do not promote to confirmed or use as a conversion basis silently. No inference is made for either current dataset. |
| UNKNOWN | No adequate evidence identifies denomination. Leave currency code unassigned/null and record unknown status. Do not use a made-up currency code or select one from symbol, country, government origin, or project preference. |

For each monetary measure or explicitly documented shared monetary scope, propose preserving **original amount/token**, parsed amount, original currency declaration if any, currency code if confirmed, currency source/reference, currency status, and the relevant source date/period with its meaning. A dataset-level currency declaration can apply to rows only when that scope is documented. Never infer that all measures/records share denomination simply because one might.

**Current evidence:** both notebooks leave currency unconfirmed. Company Financials uses `$` in all eight measure columns, including quantity. Receiver General's governmental provenance alone is not a currency declaration for its item-amount field. Therefore both retain UNKNOWN status; Units Sold is a quantity with unresolved unit semantics and must not be assigned monetary currency.

USD, CAD, BRL, and EUR illustrate an ISO-style three-letter representation **only when known and validated**; none is assigned now. A future inferred proposal would not overwrite the unconfirmed original denomination. No exchange rates, conversion dates, converted amounts, netting across currencies, or FX logic are created by cleaning. Any future conversion stage must preserve original facts and its own rate/source/date metadata separately. Evidence: I-RG conclusions; CF-C §16; CF-R compatibility limits.

## 12. Duplicate Policy

| Situation | Observed evidence | Proposed rule |
|---|---|---|
| Exact duplicate source rows | Zero in each dataset | Recheck before/after representation changes; preserve all current rows. Any future exact duplicates still require lineage/business review before removal. |
| Repeated identifier | Receiver General control number repeats 148 times after first | Preserve: grouping identifiers may repeat legitimately. Do not deduplicate on this field alone. |
| Repeated business dimensions | Company Financials five-field combination repeats 12 times | Preserve: price-dependent observations may be distinct. Do not collapse, aggregate, or choose an arbitrary first/last row. |
| Candidate-key conflict | Zero for the documented composite candidates in both extracts | Recheck with null-component counts before/after standardization. A new collision blocks publication as an unexplained rule effect; retain conflicting rows for review. |
| Apparently identical cleaned rows | Not established by EDA; trimming/parsing can change equivalence | Preserve source-row lineage and report newly merged representations. Do not claim source duplication or remove rows solely from normalized equality. |

Removal could eventually be justified only by independent evidence that the same source/business event was accidentally duplicated, an approved rule defining equivalence and survivor selection, and a complete removal/retention audit with count and amount reconciliation. No such evidence or removal rule exists now. Candidate uniqueness remains extract-specific and does not establish a database key. Evidence: I-RG §11; I-CF §11; RG-B; CF-B.

## 13. Standardization Policy

These are **proposed conventions**, not implemented schema decisions. Source-specific parsing remains separate from global output conventions.

| Area | Proposed convention | Safeguard / unresolved decision |
|---|---|---|
| Column names | Explicit, versioned one-to-one English `snake_case` map from exact source headers | Assert no collisions; preserve the original header map; never rename Sales as company revenue or RG amounts as expenses |
| Identifiers | Nullable strings read from original tokens | Preserve leading zeros/alphanumeric characters; no inferred padding, decimal stripping, or numeric modeling semantics |
| Dates | Date-only semantic representation; ISO YYYY-MM-DD serialization | Concrete Pandas date representation can be selected in implementation if round-trip checks pass; no timezone invented |
| Periods | Preserve fiscal labels separately; Company Financials monthly marker remains documented as monthly | Derived calendar periods are analysis variables, not replacements for source fields |
| Categories | Nullable strings; CF approved outer-whitespace trimming only | Retain raw text, case, accents, and internal spacing. Case folding/translations/category merging deferred. RG whitespace issues were not established; audit before proposing changes |
| Missing values | Logical null plus separate reason/status; literal None remains a category | Empty serialization must distinguish source absence from unresolved token via raw companion and status; no string 'nan' as a fabricated category |
| Money | Exact decimal parsed from source tokens, preserving source precision/sign | No assumed two-decimal currency rule; no binary float as preservation authority; no quantization or new rounding |
| Quantity | Decimal-capable Units Sold; exact nullable integer for validated RG summary counts | Dollar quantity format does not establish currency; no fractional-unit rounding |
| Currency codes | Uppercase three-letter code only under documented scope/confirmation; inferred proposal kept separate | UNKNOWN has no assigned code; no default currency or FX |
| Audit metadata | Source hash/row locator, rule IDs/version, original-to-cleaned mapping, parse/validation status | Metadata is lineage, not fabricated company or transaction identity |

Exact decimal storage/serialization must round-trip without precision loss. A decimal-preserving text export with a companion type manifest is a possible first output; typed columnar formats remain a separate decision if they require dependencies. These are output conventions, not database implementation.

## 14. Data Quality Flags

Flags are conceptual, with evidence-linked scope and neutral names. They are not implemented now and do not authorize deletion. A flag record should identify dataset, row/field when applicable, rule/version, condition, status, evidence, and intended limitation. Not assessed is distinct from false.

| Proposed flag/status | Scope and trigger | Current evidence / treatment |
|---|---|---|
| `source_missing` | Field: original source value is absent under documented parser semantics | RG missingness; preserve null, do not classify automatically as invalid |
| `unresolved_placeholder` | Field: recognized token lacks a numeric interpretation | CF 53 Discounts and 5 Profit dollar-dash tokens; retain original and null parsed value |
| `parsing_failed` | Field: nonmissing, nonplaceholder token fails the approved grammar | Zero observed unrecognized CF formats/date failures; parser-audit safeguard, not a claim of existing failures |
| `currency_unknown` | Dataset/amount scope lacking confirmed denomination | Both sources; quantity fields excluded from currency assignment |
| `currency_inferred` | Scope with later explicitly approved inferential currency evidence | Not currently populated; reserved distinction required by the currency policy |
| `source_semantics_unresolved` | Dataset/field: sign, quantity unit, grain, or definitions not established | CF units/sign/currency; RG mixed record relationships; do not fabricate row type |
| `key_conflict` | Row set: duplicate complete candidate key under proposed representation | Zero observed; key/collision validation safeguard; preserve all conflicting rows |
| `key_component_missing` | Row: a documented candidate-key component is absent | Zero observed for both candidates; future baseline-change check, not a universal required-field rule |
| `temporal_label_conflict` | CF row: parsed date disagrees with supplied month/year/name | Zero observed; do not use this flag for unresolved RG fiscal interpretation |
| `fiscal_alignment_unresolved` | RG dataset-level documentation status | Known unresolved fiscal-to-effective-date relationship; no assumed mismatch count |
| `coverage_gap_observed` | Category/period report: no row within an explicitly stated comparison interval | CF uneven combination histories; RG dates absent from extract; not proof of missing business events or a row-imputation instruction |
| `extreme_value_review_candidate` | Optional analysis-report record with measure, peer, threshold, and sample size | EDA global/peer disagreement; flag is not error/fraud/anomaly truth and does not alter data |
| `financial_relationship_review` | CF row/check: testable arithmetic residual beyond approved soft tolerance | Zero observed within EDA tolerance; report residuals separately from source-preservation failures |

Avoid a generic `missing_required_field` until a dataset-specific mandatory-field contract is approved. Do not flag all sparse optional values as required-field failures. Do not materialize ratios, growth, ML scores, or guessed fiscal/record-type labels as cleaning flags. Evidence: RG-B/M/T/X; CF-B/P/R/M/X/C.

## 15. Cleaning Rule Matrix

This consolidates the detailed rules; it does not replace their field-level conditions. Required = safe execution/publication prerequisite after approval; Recommended = useful documentary/review addition; Deferred = evidence or explicit decision still needed. No arbitrary risk score is assigned.

| Dataset | Field / Area | Issue | Rule | Action | Priority | Evidence |
|---|---|---|---|---|---|---|
| Both | Source and row lineage | Preserve facts and reproducibility | Immutable hashes/snapshots, working copy, row locator, rule versions; no row removal | PRESERVE AS-IS; VALIDATE | Required | RG-B; CF-B; final EDA checks |
| Both | Headers and raw companions | Long/space-padded labels; conversion traceability | Explicit injective header map; retained originals | STANDARDIZE; VALIDATE | Required | I-RG; I-CF; CF-P |
| RG | RG01/04/06/08/10/15 identifiers | Numeric storage vs categorical meaning | Source-token nullable strings; no float reconstruction or padding | PARSE; VALIDATE | Required | I-RG; RG-X |
| RG | RG02/09/11/13/16 codes | Optional/alphanumeric classifications | Preserve codes and nulls; no guessed mappings | KEEP; FLAG | Required | RG-D; RG-M |
| RG | RG05 effective date | Source date is text | Strict YYYY-MM-DD date with raw companion | PARSE; VALIDATE | Required | RG-B; RG-T |
| RG | RG03/07 fiscal labels | Exact alignment unresolved | Retain labels; no calendar rewrite | PRESERVE AS-IS; DEFER | Required preservation; Deferred interpretation | I-RG §9; RG-T |
| RG | RG12 amount; RG14 count | Float storage; distinct semantic measures | Exact decimal amount; count integer only after validation; retain zeros/extremes | PARSE; VALIDATE | Required | RG-D; RG-X |
| RG | Missing accounting dimensions | Selective coverage | Keep nulls, distinguish source absence and applicability uncertainty | KEEP; FLAG | Required | RG-M |
| RG | Record taxonomy/overlap | Summary/item relationships unknown | No inferred taxonomy, deduplication, or detail reconstruction | DEFER | Deferred | I-RG §4; RG-X |
| CF | CF01–04/15 categories | Outer whitespace, literal None | Audited outer trim where supported, preserve raw/case and category meaning | STANDARDIZE; VALIDATE | Required | I-CF; CF-M |
| CF | CF05–12 numeric strings | Symbol/grouping/parentheses | Strict field-scoped token parser to exact decimals; raw/status retained | PARSE; VALIDATE | Required, subject to explicit parser-rule approval | CF-P |
| CF | Discounts/Profit dash semantics | 58 unresolved tokens | Null parsed value with unresolved status, no zero imputation | FLAG; DEFER | Required preservation; Deferred interpretation | CF-M |
| CF | CF13–16 time labels | Monthly markers and uneven categories | Strict date parse and label cross-check; no gap filling | PARSE; VALIDATE | Required | CF-B; CF-D |
| CF | Company/entity | Absent | No fake company identifiers or company-period uniqueness claim | DEFER | Deferred | CF-B; CF-D |
| CF | Financial relationships | Arithmetic agreement, semantics not fully documented | Report residuals as soft checks; do not repair operands | VALIDATE; FLAG | Recommended | CF-R |
| Both | Candidate keys/duplicates | Repeated values can be valid | Compare source/cleaned candidates and collision reports; no automatic deletion | VALIDATE | Required | I-RG §11; I-CF §11 |
| Both | Currency | Denomination unconfirmed | Unknown status and unassigned code; preserve evidence; no FX | FLAG; DEFER | Required metadata; Deferred assignment/FX | I-RG conclusions; CF-C |
| Both | Extreme review | Context-sensitive screens | Retain values; optional contextual audit report | KEEP; FLAG | Required retention; Recommended review | RG-X; CF-X |
| Both | Output validity | Representation changes must preserve meaning | Count/null/key/value reconciliation and save gate | VALIDATE | Required | EDA final checks; §§18–19 of this plan |

Unresolved semantic decisions can remain explicit limitations in a row-preserving standardized export if approved; they cannot be silently resolved by the implementation. Unexpected conversion loss, changed rows, or unreviewed key collisions block publication.

## 16. What Must Not Be Cleaned Automatically

- Do not delete or cap RG large amounts/zeros, CF negative Profit, or rare categories because a statistical screen selects them. Context does not prove invalidity.
- Do not fill or discard sparse RG voucher, ledger, coding, subledger, or control fields merely because they are optional or highly missing.
- Do not turn CF dollar-dashes into zeros, even when other fields reconcile as if zero. Do not treat the category None as null.
- Do not infer currencies from `$`, country, government provenance, or the provisional BRL project setting; do not homogenize amounts with FX.
- Do not rewrite RG fiscal periods from Accounting Effective Date, fill absent dates, or treat CF first-of-month markers as individual transaction dates.
- Do not infer company identity from segment/country/product, invent balance-sheet fields, or fill category histories with artificial records.
- Do not deduplicate RG on control number alone or CF on the five-field combination without price; do not collapse mixed summary/item records.
- Do not reinterpret DR/CR as expenses/revenues, attach cash-flow signs, or treat Profit as cash receipts or verified net income.
- Do not strip alphanumeric code characters, guess padded ID widths, aggregate fractional quantities to integers, or merge categories without a reviewed collision report.
- Do not rebalance amounts to force identities, divide summary amounts into synthetic detail, or impose an unverified currency rounding convention.

Evidence: I-RG documentation review; RG-M/T/X; CF-B/P/R/M/X/C. These are safeguards against unsupported semantic changes, not assertions that all source observations have been independently verified.

## 17. Expected Cleaned Outputs

**Proposal only; no folders or files are created by this plan.** Keep the datasets separate and retain the existing raw layout. A possible future run-specific structure is:

```text
data/
  raw/                           # unchanged source files
  processed/
    <approved_run_id>/
      receiver_general_standardized.csv
      company_financials_standardized.csv
      cleaning_manifest.json
      cleaning_validation_report.json
      cleaning_review_flags.csv
```

The two standardized files would preserve **9,282 and 700 rows respectively** for the current source hashes, all 16 source fields through documented mappings, and the approved raw companions/status/lineage fields. They would not contain modeled features, guessed currencies, or joined datasets. Expected row counts are input-version checks, not filters for future source versions.

CSV is a **provisional interchange proposal**, not a typed database: identifiers must round-trip as strings, decimal text without scientific/float loss, and nulls distinguished from literal text using the companion type/null-policy manifest and statuses. Review this serialization contract before saving. If another format is selected, the same round-trip requirements apply and new dependencies require approval.

The manifest would record input paths/hashes, source/redistribution attribution as known, rule/parser versions, field/type maps, environment, null/currency policies, and unresolved assumptions. The validation report would separate hard-gate outcomes, baseline changes, and soft findings. The review flags file would retain lineage and reasons; it is not a rejected-row dump. Use a new run identifier rather than silently overwrite an earlier approved output.

## 18. Cleaning Execution Plan

Future work belongs in **`05_data_cleaning.ipynb`**, only after the user approves this plan and the implementation scope. Proposed order:

1. Record approved rules, parser assumptions, unresolved decisions, output contract, and source-version expectations. Stop and report a new material issue instead of inventing a rule.
2. Hash raw files and existing notebooks; load an immutable raw snapshot and a separate lexical/token-preserving view without modifying either source.
3. Reproduce row/column/null/key/date baselines. If the source hash or content differs, investigate the version change; do not force the old expected counts by filtering.
4. Create separate working copies with source-file/row lineage and raw companions. Apply the explicit header mapping; validate one-to-one correspondence and absence of collisions.
5. Parse source-token identifiers, dates, counts, and financial/quantity values using the approved dataset-specific rules; keep raw tokens and distinct parse statuses. Do not infer semantic values from failed tokens.
6. Apply approved CF category outer-whitespace standardization; compare before/after categories and candidate keys. Preserve literal None and report collisions. No unapproved RG value normalization.
7. Preserve source nulls; represent unresolved CF dashes as typed null-with-status. Report all changes in availability counts; no imputation, column removal, or row deletion.
8. Attach currency/semantic/provenance status at the documented scope; no default currency, inferred company, fiscal mapping, or FX.
9. Revalidate source versus represented keys, duplicate rows, date-label agreement, count integrality, and value preservation. Any unexpected loss/collision blocks output publication pending review.
10. Run approved soft CF financial checks with residuals/testability; produce coverage and optional extreme-review reports without changing values. Keep analytical ratios, growth, and ML outside cleaned base facts.
11. Reconcile per-row values, available subsets, totals, nulls, keys, dtypes, flags, and raw hashes under §19. Clearly distinguish expected unresolved semantics from hard failures.
12. Present validation results. Save only to the approved new processed location if all hard gates pass and planned unresolved states are explicitly allowed. If not, preserve raw inputs and report blockers; do not publish a misleading cleaned dataset.
13. Verify a saved-file round trip under the declared type/null contract, record output hashes, and report scope, limitations, and any deviations. Do not proceed to modeling, schema, FX, or application work.

No part of this order is executed in notebook 04. A standardization export may retain unknown currency and unresolved placeholders without claiming operational Financial Manager readiness.

## 19. Validation Requirements

The future implementation must prove the following before publication. Hard gates check preservation and declared representation; soft checks do not silently become corrective rules.

| Validation | Required proof / acceptance condition |
|---|---|
| Source immutability | Before/after hashes match for all raw files and earlier notebooks; immutable DataFrame values/types equal their snapshots. No raw writes. |
| Row lineage/counts | One output row per input row; current baselines 9,282 and 700; no undocumented deletion, insertion, aggregation, or replacement. Every output row resolves to a source row. |
| Field mapping | All 16 source columns per dataset accounted for through an injective header map and retained raw representation; no duplicate output names or lost fields. |
| RG baseline | Original missing total 29,876 with exact per-field counts; zero complete duplicates; dates parse; complete candidate key remains unique. Any discrepancy is reported, not repaired to fit. |
| CF baseline | Zero raw nulls/complete duplicates; 12 five-field repetitions, zero six-field candidate conflicts; 16 monthly labels; no invented company count. |
| Token/parse reconciliation | For every field: populated + source missing = rows; successful + unresolved placeholder + unrecognized failure = populated. CF expects 5,542 numeric interpretations, 58 unresolved placeholders, zero unrecognized formats under the approved convention. |
| Null comparison | Maintain source-null counts independently from typed availability. CF Discounts typed null count 53 and Profit 5 are explained placeholders, not dropped rows or inferred source nulls. Other approved parse failures, if any, require explicit disposition and reconciliation. |
| Identifier fidelity | Exact source-to-string correspondence and retained tokens; no rounding, leading-zero loss, accidental alphanumeric truncation, or guessed padding. Validate any representation-induced equivalence/collision. |
| Keys/duplicates | Source and represented candidate keys tested with missing components counted separately. No unreviewed new collision. Preserve repeated grouping references and the source-level duplicate report. |
| Dates/periods | Strict formats round-trip; CF date/month/year/name agree; RG fiscal labels are unchanged. No synthesized periods or claimed fiscal reconciliation. |
| Per-value numeric preservation | Exact decimal values match the approved source-token interpretation and signs/scales; no quantization or conversion of dashes to zero. Explain float-vs-decimal display differences without altering source magnitude. |
| Monetary/quantity reconciliation | Compare counts and exact sums before/after **for the same successfully parsed rows per measure**, with an independently token-parsed reference. RG direction/category subtotals and CF measure totals reconcile as representation checks. Unknown currencies make these numerical checks, not economic totals. Price fields are checked per value, not summed as business amounts. |
| Approximate EDA comparison | Recorded EDA float statistics may differ at binary precision from exact decimals. Report differences; any comparison tolerance must be explicit and cannot authorize a monetary change. Do not reuse the CF identity tolerance to permit source-value drift. |
| Soft financial checks | Report testable/untestable counts and exact residuals for the three CF relationships; compare with 700/647/695 testable EDA rows and documented tolerance. A deviation triggers investigation, not balancing or filling. |
| Dtypes/statuses | Validate identifier/category strings, date-only semantics, integer counts/labels, exact decimal values, logical nulls, and distinct statuses. Retain all 58 CF negative Profit values and both RG zeros unless a separately approved correction exists; no such correction is planned. |
| Currency | Unknown codes remain unassigned; original evidence/status/date preserved. No country/symbol/default mapping or converted values. No monetary currency assigned to Units Sold. |
| Reproducibility/round trip | Same input/rules yield equivalent rows, values, flags, and metadata (excluding declared run timestamps). Serialization reload preserves string IDs, decimals, null reasons, and literal categories. |
| Scope | No cleaned/processed output before approval and hard-gate success; no unapproved data deletion, monetary change, FX, ML, production features, schema, or application work. |

Existing clean-run expectations are baselines, not universal validity domains. Future source changes or new material quality issues must be reported separately before extending the cleaning rules. Statistical extremes and coverage gaps alone never fail the preservation gate.

## 20. Final Cleaning Plan

**Required after approval:** immutable source/snapshot protection; reproducible lineage and explicit field mapping; source-faithful identifier/date/quantity/monetary representations; audited CF whitespace treatment; null/placeholder/parse-failure separation; candidate-key and duplicate reconciliation; retained extremes and original financial meaning; unknown currency metadata; and all hard preservation/round-trip checks.

**Recommended:** category-dependent missingness reports, exact CF arithmetic residuals, coverage reports, and optional contextual extreme-review evidence with method/population recorded. These are review aids and may not mutate observations or create production features.

**Deferred decisions:** resolving RG fiscal alignment and summary/item relationships; source-token identifier conventions where ambiguous; CF dash-as-zero and sign/unit semantics beyond the explicitly approved parser convention; entity identity; exact source/redistribution provenance; currency denomination; business rounding/materiality; any actual correction, duplicate removal, imputation, or schema. No blanket deletion or fill rule is proposed.

**Multi-currency boundary:** preserve original amounts and the evidence/state of currency knowledge, accommodating multiple currencies without assuming one. Unknown remains unassigned. FX and any base-currency decision belong to a later separately approved stage.

**Unresolved dependencies do not justify fabricated facts.** A row-preserving standardized representation may explicitly carry unresolved states if approved, but it must not claim a verified company panel, cash ledger, uniform record grain, currency-comparable totals, or operational readiness. The capability assessments in notebooks 01–03 remain unchanged.

**Exact recommended next step:** review and approve this plan, explicitly including the provisional CF numeric/sign grammar, unresolved-placeholder policy, exact-decimal/source-token preservation approach, and processed-output serialization contract. Then authorize creation and execution of `05_data_cleaning.ipynb` implementing only those approved standardization, status, and validation rules. New material findings must be reported before adding rules. No implementation begins merely because this plan exists.

### Plan-Only Completion Check

This notebook consists exclusively of Markdown. No datasets were loaded into working DataFrames, transformed, exported, or cleaned to create this plan. Evidence was taken from the existing inventory/EDA documentation and saved outputs. Raw files and notebooks 01–03 were checked for unchanged hashes during plan creation. No new material data-quality issue was identified during the documentation review. All actions above remain proposed and unexecuted.